In [3]:
import sqlite3
import chromadb

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

# Перевірка SqliteSaver
conn = sqlite3.connect(":memory:", check_same_thread=False)
saver = SqliteSaver(conn)
print("✅ SqliteSaver: OK")

# Перевірка ChromaDB
client = chromadb.Client()

collection = client.get_or_create_collection("test_collection")

collection.add(
    documents=[
        "LangGraph — це фреймворк для створення агентних workflow.",
        "ChromaDB — векторна база даних для семантичного пошуку."
    ],
    ids=["doc1", "doc2"]
)

results = collection.query(
    query_texts=["векторна база даних"],
    n_results=1
)



✅ SqliteSaver: OK


In [6]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.1
)

response = llm.invoke("Відповідай одним словом: 2 + 2 = ?")

print("✅ Gemini підключено")
print("Відповідь:", response.content)

✅ Gemini підключено
Відповідь: [{'type': 'text', 'text': 'Чотири', 'extras': {'signature': 'EsEFCr4FARFNMg82juft3bMmpRosUvioJrS0+sHGzfNRENra1g9N4dkOHej4u8v4m/+muIy7fYYmDH5e8cYlmbjXAWsMsPLggUeNzdoEUrOFOaqam3VBCCEz7GggT2Y3OJIDaa4lT7tmOTNbhz2je+nAfyAlptUjUbzGq08vnPV+bRFzkL4swPUyd15OLlBxBYyxyzO95ngS1HQzPTqtzfi4s1gwJ74vNgrz+6Gr00cJGFKWA2IXeNDw0fRmw6uCkX9PAT/x4mdUZA3+GDves7XHch0ZEv2UKfhHsBxC7ebmGUuxRZIh01VYvfucHFbvr5wbqCV8C+f72AQbzU3eW0UpHGFSCMeCJB4RdAP3Gq/FbuCTNuoV8xXS0Z0s8bW4bvEKfjCuVsw4+B/orpkfrfAonoSklQWfZmVBiK/z8ioV2RN4PrXg6O9o+J1AOx3lTN/PSbrx3RES65vVij+gWB2+L/p9UDLBjHd/5Fnowfu1G243Qu/e2k5JOpWsyvAKbYYDRmOM0NTil4LOdDxAGEmba3a7bken4jcoTS2/gKnSdhowvhIpI/ODGQpPLJkrqS31rxsMSTeeDI2XOo/Dc39sQnPKLN9Ok6JPO7TX6FyLPS0gl/oAAY6LwPw4irA2/2UKwRay9HtwozfWuo4MTGwMjev1kLGA9d+KWaovvIYisQdZLXBafHqu2RxfriYMyDO8PiQ4QKbRxx7SENNz4FZirSHlp8r/B1pC5g6wUwrh7aG5qb60fuBtrfypUM39fFxQ6K31MEzeNEHPrefEsfCdfLFWtHcWffsgeh9n54KWE3MJlEPGKMcMOaqHBsB9os/X8BxODx4X/fRbMXBpsZ21cRM67FlnQyX5Kaxa0lZkKRh3m+oCOIhne0r3dxcLuI/lUd2ayJkG

In [7]:
from typing import Annotated, TypedDict, Literal
import operator

from pydantic import BaseModel, Field


# ============================================================
# 1. STRUCTURED OUTPUT ДЛЯ PLANNER
# ============================================================

class Plan(BaseModel):
    """Структурований план виконання задачі."""

    goal: str = Field(
        description="Головна ціль задачі користувача"
    )

    steps: list[str] = Field(
        description="Послідовний список конкретних кроків для досягнення цілі"
    )


# ============================================================
# 2. STRUCTURED OUTPUT ДЛЯ REPLANNER
# ============================================================

class ReplanDecision(BaseModel):
    """Рішення після виконання чергового кроку."""

    action: Literal["continue", "replan", "finish"] = Field(
        description=(
            "continue — продовжити виконання поточного плану; "
            "replan — змінити залишковий план; "
            "finish — завершити виконання"
        )
    )

    updated_steps: list[str] | None = Field(
        default=None,
        description="Оновлений список кроків, якщо action='replan'"
    )

    reasoning: str = Field(
        description="Коротке пояснення рішення replanner"
    )


# ============================================================
# 3. STATE LANGGRAPH
# ============================================================

class PlanExecuteState(TypedDict):
    """Стан Plan-and-Execute агента."""

    # Історія повідомлень.
    # operator.add дозволяє LangGraph додавати нові повідомлення,
    # а не перезаписувати весь список.
    messages: Annotated[list, operator.add]

    # План, сформований planner
    plan: list[str]

    # Номер поточного кроку, 0-indexed
    current_step: int

    # Результати вже виконаних кроків
    results: list[str]

    # Ознака завершення задачі
    completed: bool


print("✅ Plan створено")
print("✅ ReplanDecision створено")
print("✅ PlanExecuteState створено")
print("🎯 Structured outputs та State готові")

✅ Plan створено
✅ ReplanDecision створено
✅ PlanExecuteState створено
🎯 Structured outputs та State готові


In [8]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool


# ============================================================
# PYDANTIC-СХЕМИ ДЛЯ TOOLS
# ============================================================

class CalculatorInput(BaseModel):
    expression: str = Field(
        description="Математичний вираз, наприклад: 120 * 3 + 45"
    )


class WeatherInput(BaseModel):
    city: str = Field(
        min_length=2,
        description="Назва міста"
    )


class CurrencyInput(BaseModel):
    amount: float = Field(
        gt=0,
        description="Сума для конвертації"
    )
    rate: float = Field(
        gt=0,
        description="Курс валют"
    )


class HotelSearchInput(BaseModel):
    city: str = Field(
        min_length=2,
        description="Місто для пошуку готелю"
    )
    max_price: float = Field(
        gt=0,
        description="Максимальна ціна за ніч у EUR"
    )


# ============================================================
# РЕАЛІЗАЦІЯ TOOLS
# ============================================================

def calculate_func(expression: str) -> str:
    """Безпечно виконує прості математичні обчислення."""
    try:
        allowed_chars = set("0123456789+-*/(). ")
        if not set(expression).issubset(allowed_chars):
            return "Помилка: вираз містить недозволені символи"

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception as e:
        return f"Помилка обчислення: {e}"


def weather_func(city: str) -> str:
    """Демонстраційний tool погоди."""
    return (
        f"Демонстраційний прогноз для {city}: "
        f"температура +22°C, без опадів."
    )


def currency_func(amount: float, rate: float) -> str:
    """Конвертація суми за заданим курсом."""
    result = amount * rate
    return f"{amount} × {rate} = {result:.2f}"


def hotel_search_func(city: str, max_price: float) -> str:
    """Демонстраційний пошук готелю."""
    return (
        f"Знайдено варіант у місті {city}: "
        f"Demo Hotel — {max_price * 0.8:.2f} EUR/ніч."
    )


# ============================================================
# СТВОРЕННЯ LANGCHAIN TOOLS
# ============================================================

calculator = StructuredTool.from_function(
    func=calculate_func,
    name="calculator",
    description=(
        "Використовуйте для математичних обчислень. "
        "Не використовуйте для довідкової інформації."
    ),
    args_schema=CalculatorInput
)


get_weather = StructuredTool.from_function(
    func=weather_func,
    name="get_weather",
    description=(
        "Використовуйте для отримання демонстраційного прогнозу погоди."
    ),
    args_schema=WeatherInput
)


convert_currency = StructuredTool.from_function(
    func=currency_func,
    name="convert_currency",
    description=(
        "Використовуйте для конвертації суми за відомим курсом."
    ),
    args_schema=CurrencyInput
)


search_hotels = StructuredTool.from_function(
    func=hotel_search_func,
    name="search_hotels",
    description=(
        "Використовуйте для пошуку готелю у заданому місті "
        "з обмеженням максимальної ціни."
    ),
    args_schema=HotelSearchInput
)


tools = [
    calculator,
    get_weather,
    convert_currency,
    search_hotels,
]

tools_by_name = {tool.name: tool for tool in tools}


print("✅ Tools створено:")
for tool in tools:
    print(" -", tool.name)

✅ Tools створено:
 - calculator
 - get_weather
 - convert_currency
 - search_hotels


In [9]:
print("Calculator:")
print(calculator.invoke({"expression": "120 * 3 + 45"}))

print("\nWeather:")
print(get_weather.invoke({"city": "Paris"}))

print("\nCurrency:")
print(convert_currency.invoke({
    "amount": 100,
    "rate": 1.08
}))

print("\nHotel:")
print(search_hotels.invoke({
    "city": "Paris",
    "max_price": 120
}))

Calculator:
405

Weather:
Демонстраційний прогноз для Paris: температура +22°C, без опадів.

Currency:
100.0 × 1.08 = 108.00

Hotel:
Знайдено варіант у місті Paris: Demo Hotel — 96.00 EUR/ніч.


In [10]:
# ============================================================
# STRUCTURED LLM ДЛЯ PLANNER ТА REPLANNER
# ============================================================

planner_llm = llm.with_structured_output(Plan)
replanner_llm = llm.with_structured_output(ReplanDecision)

print("✅ planner_llm створено")
print("✅ replanner_llm створено")

✅ planner_llm створено
✅ replanner_llm створено


In [11]:
# ============================================================
# ОДНОРАЗОВИЙ ТЕСТ STRUCTURED OUTPUT
# ============================================================

test_plan = planner_llm.invoke(
    """
    Створи короткий план для задачі:
    Організувати поїздку до Парижа на 3 дні.

    Використай 3 конкретні послідовні кроки.
    """
)

print("✅ Structured output працює")
print("Тип:", type(test_plan))
print("Goal:", test_plan.goal)

print("Steps:")
for i, step in enumerate(test_plan.steps, 1):
    print(f"{i}. {step}")

✅ Structured output працює
Тип: <class '__main__.Plan'>
Goal: Організувати поїздку до Парижа на 3 дні
Steps:
1. Забронювати квитки на транспорт та житло в Парижі
2. Скласти щоденний маршрут відвідування пам'яток та придбати вхідні квитки онлайн
3. Підготувати документи, страховку та спланувати бюджет поїздки


In [12]:
from langchain_core.messages import HumanMessage, AIMessage


# ============================================================
# ДОПОМІЖНА ФУНКЦІЯ
# ============================================================

def message_to_text(message):
    """Перетворює content повідомлення Gemini у звичайний текст."""
    content = getattr(message, "content", message)

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []

        for item in content:
            if isinstance(item, dict):
                text = item.get("text")
                if text:
                    parts.append(text)
            else:
                parts.append(str(item))

        return " ".join(parts)

    return str(content)


# ============================================================
# PLANNER NODE
# ============================================================

def planner_node(state: PlanExecuteState) -> dict:
    """Створює початковий план виконання задачі."""

    user_message = message_to_text(state["messages"][0])

    prompt = f"""
    Ти planner у Plan-and-Execute агенті.

    Задача користувача:
    {user_message}

    Створи короткий план із 2-4 конкретних послідовних кроків.

    Доступні інструменти:
    - calculator — математичні обчислення
    - get_weather — отримання погоди
    - convert_currency — конвертація за заданим курсом
    - search_hotels — пошук готелю

    Якщо крок потребує інструмента, явно вкажи його назву
    у тексті кроку.

    Не додавай зайвих кроків.
    """

    plan_result = planner_llm.invoke(prompt)

    return {
        "plan": plan_result.steps,
        "current_step": 0,
        "results": [],
        "completed": False,
        "messages": [
            AIMessage(
                content="План створено:\n" +
                "\n".join(
                    f"{i+1}. {step}"
                    for i, step in enumerate(plan_result.steps)
                )
            )
        ]
    }


# ============================================================
# EXECUTOR NODE
# ============================================================

def executor_node(state: PlanExecuteState) -> dict:
    """Виконує один поточний крок плану."""

    step_index = state["current_step"]

    if step_index >= len(state["plan"]):
        return {"completed": True}

    current_step = state["plan"][step_index]

    # LLM отримує доступ до наших tools
    executor_llm = llm.bind_tools(tools)

    response = executor_llm.invoke(
        f"""
        Виконай ОДИН крок плану:

        {current_step}

        Попередні результати:
        {state["results"]}

        Якщо для кроку потрібен доступний tool —
        обов'язково виклич відповідний tool.
        """
    )

    result_text = message_to_text(response)

    # Якщо Gemini вирішив викликати tool
    if response.tool_calls:

        tool_results = []

        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            tool_function = tools_by_name.get(tool_name)

            if tool_function:
                tool_result = tool_function.invoke(tool_args)

                tool_results.append(
                    f"{tool_name}: {tool_result}"
                )

        if tool_results:
            result_text = "\n".join(tool_results)

    new_results = list(state["results"])
    new_results.append(
        f"Крок {step_index + 1}: {result_text}"
    )

    return {
        "current_step": step_index + 1,
        "results": new_results,
        "messages": [
            AIMessage(
                content=(
                    f"Виконано крок {step_index + 1}: "
                    f"{result_text}"
                )
            )
        ]
    }


# ============================================================
# REPLANNER NODE
# ============================================================

def replanner_node(state: PlanExecuteState) -> dict:
    """
    Аналізує результат і вирішує:
    continue / replan / finish.
    """

    # Якщо всі кроки вже виконані — LLM зайвий раз НЕ викликаємо.
    if state["current_step"] >= len(state["plan"]):
        return {
            "completed": True,
            "messages": [
                AIMessage(
                    content="Усі кроки плану виконані."
                )
            ]
        }

    remaining_steps = state["plan"][state["current_step"]:]

    prompt = f"""
    Ти replanner у Plan-and-Execute агенті.

    Повний план:
    {state["plan"]}

    Виконано кроків:
    {state["current_step"]}

    Отримані результати:
    {state["results"]}

    Залишкові кроки:
    {remaining_steps}

    Вибери:
    - continue — якщо план залишається актуальним;
    - replan — тільки якщо залишковий план реально треба змінити;
    - finish — якщо ціль вже досягнута раніше.

    Не переплановуй без необхідності.
    """

    decision = replanner_llm.invoke(prompt)

    if decision.action == "finish":
        return {
            "completed": True,
            "messages": [
                AIMessage(
                    content=f"Replanner: finish — {decision.reasoning}"
                )
            ]
        }

    if decision.action == "replan" and decision.updated_steps:
        return {
            "plan": decision.updated_steps,
            "current_step": 0,
            "messages": [
                AIMessage(
                    content=f"План змінено: {decision.reasoning}"
                )
            ]
        }

    return {
        "messages": [
            AIMessage(
                content=f"Replanner: continue — {decision.reasoning}"
            )
        ]
    }


print("✅ planner_node готовий")
print("✅ executor_node готовий")
print("✅ replanner_node готовий")
print("💰 Gemini API при виконанні цієї комірки НЕ викликався")

✅ planner_node готовий
✅ executor_node готовий
✅ replanner_node готовий
💰 Gemini API при виконанні цієї комірки НЕ викликався


In [13]:
from langgraph.graph import StateGraph, START, END


# ============================================================
# ROUTER ПІСЛЯ REPLANNER
# ============================================================

def should_continue(state: PlanExecuteState):
    """
    Якщо completed=True — завершуємо граф.
    Інакше повертаємося до executor.
    """
    if state.get("completed", False):
        return "end"

    return "executor"


# ============================================================
# СТВОРЕННЯ ГРАФА
# ============================================================

graph = StateGraph(PlanExecuteState)

# Додаємо три основні вузли
graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)
graph.add_node("replanner", replanner_node)

# Основний потік
graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "replanner")

# Після replanner:
# або наступний executor,
# або завершення графа
graph.add_conditional_edges(
    "replanner",
    should_continue,
    {
        "executor": "executor",
        "end": END,
    }
)

# Поки що компілюємо БЕЗ persistence.
# SqliteSaver додамо окремо в Завданні 2.
app = graph.compile()


print("✅ LangGraph створено")
print("✅ START → planner → executor → replanner")
print("✅ replanner → executor / END")
print("💰 Gemini API не викликався")

✅ LangGraph створено
✅ START → planner → executor → replanner
✅ replanner → executor / END
💰 Gemini API не викликався


In [14]:
from langchain_core.messages import HumanMessage


# ============================================================
# КОРОТКИЙ ТЕСТ PLAN-AND-EXECUTE
# ============================================================

initial_state = {
    "messages": [
        HumanMessage(
            content=(
                "Порахуй 120 * 3 + 45 і потім "
                "конвертуй результат за курсом 1.08."
            )
        )
    ],
    "plan": [],
    "current_step": 0,
    "results": [],
    "completed": False,
}


result = app.invoke(initial_state)


print("✅ Граф завершив роботу")
print("\nПлан:")
for i, step in enumerate(result["plan"], 1):
    print(f"{i}. {step}")

print("\nРезультати:")
for item in result["results"]:
    print("-", item)

print("\nCompleted:", result["completed"])

✅ Граф завершив роботу

План:
1. Використати інструмент calculator для обчислення виразу 120 * 3 + 45
2. Використати інструмент convert_currency для конвертації отриманого результату за курсом 1.08

Результати:
- Крок 1: calculator: 405
- Крок 2: convert_currency: 405.0 × 1.08 = 437.40

Completed: True


In [15]:
import sqlite3
import os

from langgraph.checkpoint.sqlite import SqliteSaver


# ============================================================
# SQLITE CHECKPOINTER
# ============================================================

DB_PATH = "agent_state.db"

# ФАЙЛОВА база — не :memory:
conn = sqlite3.connect(
    DB_PATH,
    check_same_thread=False
)

saver = SqliteSaver(conn)

# Компілюємо той самий граф, але вже з persistence
app_memory = graph.compile(
    checkpointer=saver
)


print("✅ SqliteSaver підключено")
print("✅ Граф скомпільовано з checkpointer")
print("Файл БД:", DB_PATH)
print("Файл існує:", os.path.exists(DB_PATH))
print("💰 Gemini API не викликався")

✅ SqliteSaver підключено
✅ Граф скомпільовано з checkpointer
Файл БД: agent_state.db
Файл існує: True
💰 Gemini API не викликався


In [16]:
# ============================================================
# ГРАФ ДЛЯ ДЕМОНСТРАЦІЇ PERSISTENCE
# Зупиняємося після першого executor
# ============================================================

app_pause = graph.compile(
    checkpointer=saver,
    interrupt_after=["executor"]
)

config_persist = {
    "configurable": {
        "thread_id": "persistence-demo-001"
    }
}

persist_initial_state = {
    "messages": [
        HumanMessage(
            content=(
                "Порахуй 50 * 4, а потім "
                "конвертуй результат за курсом 1.1."
            )
        )
    ],
    "plan": [],
    "current_step": 0,
    "results": [],
    "completed": False,
}


# Запускаємо граф.
# Він виконає planner + ПЕРШИЙ executor і зупиниться.
partial_result = app_pause.invoke(
    persist_initial_state,
    config=config_persist
)


print("⏸️ Граф зупинено після першого executor")
print("Thread ID:", config_persist["configurable"]["thread_id"])

print("\nПоточний крок:")
print(partial_result["current_step"])

print("\nПлан:")
for i, step in enumerate(partial_result["plan"], 1):
    print(f"{i}. {step}")

print("\nРезультати на момент зупинки:")
for item in partial_result["results"]:
    print("-", item)

print("\nCompleted:", partial_result["completed"])

⏸️ Граф зупинено після першого executor
Thread ID: persistence-demo-001

Поточний крок:
1

План:
1. Використати інструмент calculator для обчислення 50 * 4
2. Використати інструмент convert_currency для конвертації отриманого результату за курсом 1.1

Результати на момент зупинки:
- Крок 1: calculator: 200

Completed: False


In [17]:
# ============================================================
# ЧИТАЄМО ЗБЕРЕЖЕНИЙ СТАН ІЗ SQLITE
# ============================================================

restored_state = app_pause.get_state(config_persist)

print("✅ Стан відновлено з agent_state.db")
print("Thread ID:", config_persist["configurable"]["thread_id"])

print("\nПоточний крок:")
print(restored_state.values["current_step"])

print("\nПлан:")
for i, step in enumerate(restored_state.values["plan"], 1):
    print(f"{i}. {step}")

print("\nЗбережені результати:")
for item in restored_state.values["results"]:
    print("-", item)

print("\nCompleted:")
print(restored_state.values["completed"])

print("\nНаступний вузол графа:")
print(restored_state.next)

print("\n💰 Gemini API не викликався")

✅ Стан відновлено з agent_state.db
Thread ID: persistence-demo-001

Поточний крок:
1

План:
1. Використати інструмент calculator для обчислення 50 * 4
2. Використати інструмент convert_currency для конвертації отриманого результату за курсом 1.1

Збережені результати:
- Крок 1: calculator: 200

Completed:
False

Наступний вузол графа:
('replanner',)

💰 Gemini API не викликався


In [18]:
# ============================================================
# ІМІТАЦІЯ ПЕРЕЗАПУСКУ ПРОЦЕСУ
# ============================================================

# Закриваємо старе з'єднання з SQLite
conn.close()

print("🔄 Старе з'єднання закрито — імітуємо restart")


# ============================================================
# НОВЕ ПІДКЛЮЧЕННЯ ДО ТОГО САМОГО agent_state.db
# ============================================================

new_conn = sqlite3.connect(
    "agent_state.db",
    check_same_thread=False
)

new_saver = SqliteSaver(new_conn)

# Ніби програма була запущена заново
app_restored = graph.compile(
    checkpointer=new_saver
)


# Той самий thread_id
same_config = {
    "configurable": {
        "thread_id": "persistence-demo-001"
    }
}


# Читаємо стан уже через НОВИЙ SqliteSaver
state_after_restart = app_restored.get_state(same_config)


print("\n✅ Новий процес підключився до agent_state.db")
print("✅ Сесію відновлено після restart")
print("Thread ID:", same_config["configurable"]["thread_id"])

print("\nПоточний крок:")
print(state_after_restart.values["current_step"])

print("\nЗбережені результати:")
for item in state_after_restart.values["results"]:
    print("-", item)

print("\nCompleted:")
print(state_after_restart.values["completed"])

print("\nНаступний вузол:")
print(state_after_restart.next)

print("\n💰 Gemini API не викликався")

🔄 Старе з'єднання закрито — імітуємо restart

✅ Новий процес підключився до agent_state.db
✅ Сесію відновлено після restart
Thread ID: persistence-demo-001

Поточний крок:
1

Збережені результати:
- Крок 1: calculator: 200

Completed:
False

Наступний вузол:
('replanner',)

💰 Gemini API не викликався


In [19]:
# ============================================================
# ПРОДОВЖЕННЯ З ТОГО САМОГО CHECKPOINT
# ============================================================

continued_result = app_restored.invoke(
    None,
    config=same_config
)

print("▶️ Сесію продовжено після restart")
print("Thread ID:", same_config["configurable"]["thread_id"])

print("\nПоточний крок:")
print(continued_result["current_step"])

print("\nРезультати:")
for item in continued_result["results"]:
    print("-", item)

print("\nCompleted:")
print(continued_result["completed"])

▶️ Сесію продовжено після restart
Thread ID: persistence-demo-001

Поточний крок:
2

Результати:
- Крок 1: calculator: 200
- Крок 2: convert_currency: 200.0 × 1.1 = 220.00

Completed:
True


In [20]:
app_restored.invoke(None, config=same_config)

{'messages': [HumanMessage(content='Порахуй 50 * 4, а потім конвертуй результат за курсом 1.1.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='План створено:\n1. Використати інструмент calculator для обчислення 50 * 4\n2. Використати інструмент convert_currency для конвертації отриманого результату за курсом 1.1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='Виконано крок 1: calculator: 200', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='Replanner: continue — Перший крок успішно виконано, отримано результат 200. Залишковий крок залишається актуальним для виконання конвертації.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='Виконано крок 2: convert_currency: 200.0 × 1.1 = 220.00', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='Усі кроки п

In [21]:
# ============================================================
# ДЕМОНСТРАЦІЯ НЕЗАЛЕЖНОСТІ THREAD_ID
# ============================================================

# Перша сесія — наша вже завершена
config_session_1 = {
    "configurable": {
        "thread_id": "persistence-demo-001"
    }
}

# Друга сесія — новий thread_id
config_session_2 = {
    "configurable": {
        "thread_id": "persistence-demo-002"
    }
}


state_session_1 = app_restored.get_state(config_session_1)
state_session_2 = app_restored.get_state(config_session_2)


print("========== SESSION 1 ==========")
print("Thread ID: persistence-demo-001")
print("Current step:", state_session_1.values.get("current_step"))
print("Results:", state_session_1.values.get("results"))
print("Completed:", state_session_1.values.get("completed"))


print("\n========== SESSION 2 ==========")
print("Thread ID: persistence-demo-002")

if state_session_2.values:
    print("Current step:", state_session_2.values.get("current_step"))
    print("Results:", state_session_2.values.get("results"))
    print("Completed:", state_session_2.values.get("completed"))
else:
    print("Стан порожній — це нова незалежна сесія.")


print("\n✅ Різні thread_id мають незалежний стан")
print("💰 Gemini API не викликався")


========== SESSION 1 ==========
Thread ID: persistence-demo-001
Current step: 2
Results: ['Крок 1: calculator: 200', 'Крок 2: convert_currency: 200.0 × 1.1 = 220.00']
Completed: True

========== SESSION 2 ==========
Thread ID: persistence-demo-002
Стан порожній — це нова незалежна сесія.

✅ Різні thread_id мають незалежний стан
💰 Gemini API не викликався


In [22]:
import chromadb
import os

# ============================================================
# PERSISTENT CHROMADB
# ============================================================

CHROMA_PATH = "chroma_db"

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

knowledge_base = chroma_client.get_or_create_collection(
    name="travel_knowledge"
)


# ============================================================
# БАЗА ЗНАНЬ — 10 ДОКУМЕНТІВ
# ============================================================

documents = [
    (
        "Для короткострокових туристичних поїздок громадяни України "
        "можуть користуватися безвізовим режимом у Шенгенській зоні "
        "за наявності біометричного паспорта. Загальний строк перебування "
        "не повинен перевищувати 90 днів протягом 180-денного періоду."
    ),

    (
        "Париж є столицею Франції та одним із найбільших туристичних "
        "центрів Європи. Основні пам'ятки міста: Ейфелева вежа, Лувр, "
        "Тріумфальна арка, Монмартр та собор Нотр-Дам."
    ),

    (
        "Паризьке метро є одним із найзручніших способів пересування "
        "містом. Метро має велику мережу станцій, тому більшість "
        "туристичних районів легко доступні громадським транспортом."
    ),

    (
        "Для відвідування популярних туристичних об'єктів Парижа "
        "рекомендується купувати квитки онлайн заздалегідь. "
        "Це особливо актуально для Ейфелевої вежі та Лувру."
    ),

    (
        "Лувр є одним із найбільших музеїв світу. Для повноцінного "
        "відвідування музею варто запланувати декілька годин та "
        "заздалегідь визначити експозиції, які найбільше цікавлять."
    ),

    (
        "Найбільш комфортними сезонами для туристичної поїздки до Парижа "
        "вважаються весна та початок осені. У цей період температура "
        "зазвичай комфортна для тривалих прогулянок містом."
    ),

    (
        "У туристичних районах Парижа слід уважно стежити за особистими "
        "речами. Кишенькові крадіжки найчастіше трапляються у метро, "
        "на вокзалах та біля популярних пам'яток."
    ),

    (
        "Бюджет туристичної поїздки залежить від категорії житла, "
        "харчування та активностей. Для економії варто бронювати "
        "проживання завчасно та користуватися громадським транспортом."
    ),

    (
        "При виборі готелю в Парижі важливо враховувати не лише ціну, "
        "а й розташування відносно станцій метро. Готель біля метро "
        "часто дозволяє суттєво скоротити час на пересування."
    ),

    (
        "Для туристичної поїздки рекомендується мати медичне страхування, "
        "копії важливих документів та резервний спосіб оплати. "
        "Банківські картки широко приймаються у Франції."
    ),
]

doc_ids = [f"travel_doc_{i}" for i in range(len(documents))]


# Щоб повторний запуск комірки не створював дублікати
existing_ids = knowledge_base.get()["ids"]

new_documents = []
new_ids = []

for doc_id, doc in zip(doc_ids, documents):
    if doc_id not in existing_ids:
        new_ids.append(doc_id)
        new_documents.append(doc)


if new_documents:
    knowledge_base.add(
        documents=new_documents,
        ids=new_ids
    )


print("✅ Persistent ChromaDB створено")
print("Шлях:", CHROMA_PATH)
print("Кількість документів:", knowledge_base.count())
print("💰 Gemini API не викликався")

✅ Persistent ChromaDB створено
Шлях: chroma_db
Кількість документів: 10
💰 Gemini API не викликався


In [23]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool


# ============================================================
# PYDANTIC-СХЕМА ДЛЯ RAG TOOL
# ============================================================

class KnowledgeSearchInput(BaseModel):
    query: str = Field(
        min_length=3,
        description="Пошуковий запит до бази знань"
    )


# ============================================================
# ФУНКЦІЯ ПОШУКУ В CHROMADB
# ============================================================

def search_knowledge_func(query: str) -> str:
    """
    Шукає релевантну довідкову інформацію у базі знань.
    Повертає топ-3 документи.
    """

    results = knowledge_base.query(
        query_texts=[query],
        n_results=3
    )

    docs = results["documents"][0]

    if not docs:
        return "У базі знань релевантної інформації не знайдено."

    formatted = []

    for i, doc in enumerate(docs, 1):
        formatted.append(f"Документ {i}: {doc}")

    return "\n\n".join(formatted)


# ============================================================
# LANGCHAIN TOOL
# ============================================================

search_knowledge = StructuredTool.from_function(
    func=search_knowledge_func,
    name="search_knowledge",
    description=(
        "Використовуйте цей інструмент для пошуку довідкової інформації, "
        "фактів, правил, рекомендацій та знань про туристичні поїздки "
        "і Париж у локальній базі знань. "
        "НЕ використовуйте його для математичних обчислень, "
        "конвертації валют або виконання дій."
    ),
    args_schema=KnowledgeSearchInput
)

print("✅ search_knowledge tool створено")
print("💰 Gemini API не викликався")

✅ search_knowledge tool створено
💰 Gemini API не викликався


In [24]:
# ============================================================
# ТЕСТ RAG БЕЗ LLM
# ============================================================

rag_test = search_knowledge.invoke({
    "query": "Які правила перебування українців у Шенгенській зоні?"
})

print("✅ RAG search працює")
print()
print(rag_test)

print("\n💰 Gemini API не викликався")

✅ RAG search працює

Документ 1: При виборі готелю в Парижі важливо враховувати не лише ціну, а й розташування відносно станцій метро. Готель біля метро часто дозволяє суттєво скоротити час на пересування.

Документ 2: Бюджет туристичної поїздки залежить від категорії житла, харчування та активностей. Для економії варто бронювати проживання завчасно та користуватися громадським транспортом.

Документ 3: Лувр є одним із найбільших музеїв світу. Для повноцінного відвідування музею варто запланувати декілька годин та заздалегідь визначити експозиції, які найбільше цікавлять.

💰 Gemini API не викликався


In [25]:
import re


# ============================================================
# ПОКРАЩЕНИЙ HYBRID RAG SEARCH
# ChromaDB semantic search + keyword reranking
# ============================================================

def normalize_words(text: str) -> set[str]:
    """Проста нормалізація слів для keyword matching."""
    words = re.findall(r"[а-яА-ЯіІїЇєЄґҐa-zA-Z0-9]+", text.lower())

    # Відкидаємо дуже короткі слова
    return {word for word in words if len(word) >= 4}


def search_knowledge_func(query: str) -> str:
    """
    Hybrid search:
    1. ChromaDB виконує semantic search.
    2. Результати додатково ранжуються за збігом ключових слів.
    """

    # Беремо всі документи з ChromaDB.
    # База маленька (10 документів), тому це дешево.
    results = knowledge_base.query(
        query_texts=[query],
        n_results=knowledge_base.count(),
        include=["documents", "distances"]
    )

    docs = results["documents"][0]
    distances = results["distances"][0]

    query_words = normalize_words(query)

    ranked = []

    for doc, distance in zip(docs, distances):

        doc_words = normalize_words(doc)

        # Кількість спільних слів
        keyword_matches = len(query_words.intersection(doc_words))

        # Chroma distance: менше = краще
        semantic_score = 1 / (1 + distance)

        # Keyword match має більшу вагу,
        # щоб українські запити працювали стабільніше.
        total_score = keyword_matches * 2 + semantic_score

        ranked.append(
            (total_score, keyword_matches, doc)
        )

    ranked.sort(
        key=lambda x: x[0],
        reverse=True
    )

    top_docs = ranked[:3]

    formatted = []

    for i, (_, matches, doc) in enumerate(top_docs, 1):
        formatted.append(
            f"Документ {i}: {doc}"
        )

    return "\n\n".join(formatted)


# Пересоздаємо LangChain tool з покращеною функцією
search_knowledge = StructuredTool.from_function(
    func=search_knowledge_func,
    name="search_knowledge",
    description=(
        "Шукає факти, правила, рекомендації та довідкову інформацію "
        "у локальній туристичній базі знань ChromaDB. "
        "Використовуй цей tool для питань про Париж, подорожі, "
        "Шенгенську зону, безвіз, безпеку, транспорт і туристичні правила. "
        "НЕ використовуй для математичних обчислень або конвертації валют."
    ),
    args_schema=KnowledgeSearchInput
)

print("✅ Hybrid search_knowledge створено")
print("💰 Gemini API не викликався")

✅ Hybrid search_knowledge створено
💰 Gemini API не викликався


In [26]:
rag_test = search_knowledge.invoke({
    "query": "Які правила перебування українців у Шенгенській зоні?"
})

print("✅ Hybrid RAG result:\n")
print(rag_test)

print("\n💰 Gemini API не викликався")

✅ Hybrid RAG result:

Документ 1: Для короткострокових туристичних поїздок громадяни України можуть користуватися безвізовим режимом у Шенгенській зоні за наявності біометричного паспорта. Загальний строк перебування не повинен перевищувати 90 днів протягом 180-денного періоду.

Документ 2: При виборі готелю в Парижі важливо враховувати не лише ціну, а й розташування відносно станцій метро. Готель біля метро часто дозволяє суттєво скоротити час на пересування.

Документ 3: Бюджет туристичної поїздки залежить від категорії житла, харчування та активностей. Для економії варто бронювати проживання завчасно та користуватися громадським транспортом.

💰 Gemini API не викликався


In [27]:
# ============================================================
# ДОДАЄМО RAG TOOL ДО АГЕНТА
# ============================================================

# Захист від дублювання при повторному запуску комірки
if not any(tool.name == "search_knowledge" for tool in tools):
    tools.append(search_knowledge)

tools_by_name = {
    tool.name: tool
    for tool in tools
}

print("✅ Актуальні tools агента:")

for tool in tools:
    print(" -", tool.name)

print("\n💰 Gemini API не викликався")

✅ Актуальні tools агента:
 - calculator
 - get_weather
 - convert_currency
 - search_hotels
 - search_knowledge

💰 Gemini API не викликався


In [28]:
# ============================================================
# AGENTIC RAG — АВТОНОМНИЙ ВИБІР TOOL
# ============================================================

agentic_llm = llm.bind_tools(tools)


def show_tool_choice(question: str):
    """Просимо LLM самостійно вибрати потрібний tool."""

    response = agentic_llm.invoke(
        f"""
        Виконай задачу користувача.

        Самостійно виріши, який із доступних tools потрібен.
        Якщо потрібен tool — виклич його, а не відповідай навмання.

        Задача:
        {question}
        """
    )

    print("Запит:", question)

    if response.tool_calls:
        for tc in response.tool_calls:
            print("✅ Агент вибрав tool:", tc["name"])
            print("Аргументи:", tc["args"])

            tool_fn = tools_by_name[tc["name"]]
            tool_result = tool_fn.invoke(tc["args"])

            print("Результат tool:")
            print(tool_result)
    else:
        print("⚠️ Tool не викликано")
        print("Відповідь:", message_to_text(response))

    print("-" * 70)


# ------------------------------------------------------------
# ТЕСТ 1 — має вибрати RAG
# ------------------------------------------------------------

show_tool_choice(
    "Які правила короткострокового перебування громадян України "
    "у Шенгенській зоні?"
)


# ------------------------------------------------------------
# ТЕСТ 2 — має вибрати CALCULATOR
# ------------------------------------------------------------

show_tool_choice(
    "Порахуй 250 * 4 + 100"
)

Запит: Які правила короткострокового перебування громадян України у Шенгенській зоні?
✅ Агент вибрав tool: search_knowledge
Аргументи: {'query': 'правила короткострокового перебування громадян України у Шенгенській зоні безвіз'}
Результат tool:
Документ 1: Для короткострокових туристичних поїздок громадяни України можуть користуватися безвізовим режимом у Шенгенській зоні за наявності біометричного паспорта. Загальний строк перебування не повинен перевищувати 90 днів протягом 180-денного періоду.

Документ 2: При виборі готелю в Парижі важливо враховувати не лише ціну, а й розташування відносно станцій метро. Готель біля метро часто дозволяє суттєво скоротити час на пересування.

Документ 3: Бюджет туристичної поїздки залежить від категорії житла, харчування та активностей. Для економії варто бронювати проживання завчасно та користуватися громадським транспортом.
----------------------------------------------------------------------
Запит: Порахуй 250 * 4 + 100
✅ Агент вибрав tool: cal

In [29]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool


# ============================================================
# PYDANTIC-СХЕМА ДЛЯ РИЗИКОВОГО TOOL
# ============================================================

class BookHotelInput(BaseModel):
    hotel_name: str = Field(
        min_length=2,
        description="Назва готелю"
    )

    check_in: str = Field(
        description="Дата заїзду у форматі YYYY-MM-DD"
    )

    nights: int = Field(
        gt=0,
        le=30,
        description="Кількість ночей"
    )

    total_cost: float = Field(
        gt=0,
        description="Загальна вартість бронювання у EUR"
    )


# ============================================================
# РИЗИКОВА ДІЯ
# ============================================================

def book_hotel_func(
    hotel_name: str,
    check_in: str,
    nights: int,
    total_cost: float
) -> str:
    """
    Демонстраційне бронювання готелю.
    У реальній системі тут міг би бути API booking-сервісу.
    """

    return (
        f"ЗАБРОНЬОВАНО: {hotel_name}, "
        f"заїзд {check_in}, "
        f"{nights} ночей, "
        f"вартість {total_cost:.2f} EUR"
    )


book_hotel = StructuredTool.from_function(
    func=book_hotel_func,
    name="book_hotel",
    description=(
        "РИЗИКОВА ДІЯ. Використовується для остаточного бронювання готелю. "
        "Перед виконанням обов'язково потрібне підтвердження людини."
    ),
    args_schema=BookHotelInput
)


print("✅ Ризиковий tool book_hotel створено")
print("💰 Gemini API не викликався")

✅ Ризиковий tool book_hotel створено
💰 Gemini API не викликався


In [30]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt


# ============================================================
# STATE ДЛЯ HITL DEMO
# ============================================================

class HITLState(TypedDict):
    hotel_name: str
    check_in: str
    nights: int
    total_cost: float
    result: str
    approved: bool


# ============================================================
# HITL NODE
# ============================================================

def booking_with_approval(state: HITLState):
    """
    Зупиняє граф перед ризиковою дією
    та чекає рішення людини.
    """

    approval = interrupt({
        "action": "book_hotel",
        "message": "Підтвердіть бронювання готелю",
        "details": {
            "hotel_name": state["hotel_name"],
            "check_in": state["check_in"],
            "nights": state["nights"],
            "total_cost": state["total_cost"],
        }
    })

    # APPROVE
    if isinstance(approval, dict) and approval.get("approved") is True:

        result = book_hotel.invoke({
            "hotel_name": state["hotel_name"],
            "check_in": state["check_in"],
            "nights": state["nights"],
            "total_cost": state["total_cost"],
        })

        return {
            "approved": True,
            "result": result,
        }

    # REJECT
    reason = ""

    if isinstance(approval, dict):
        reason = approval.get("reason", "")

    return {
        "approved": False,
        "result": f"Бронювання ВІДХИЛЕНО. Причина: {reason}",
    }


# ============================================================
# HITL GRAPH
# ============================================================

hitl_graph = StateGraph(HITLState)

hitl_graph.add_node(
    "booking_with_approval",
    booking_with_approval
)

hitl_graph.add_edge(
    START,
    "booking_with_approval"
)

hitl_graph.add_edge(
    "booking_with_approval",
    END
)


# Використовуємо наш файловий SQLite checkpointer
app_hitl = hitl_graph.compile(
    checkpointer=new_saver
)


print("✅ HITL graph створено")
print("✅ interrupt() підключено")
print("✅ SqliteSaver використовується")
print("💰 Gemini API не викликався")

✅ HITL graph створено
✅ interrupt() підключено
✅ SqliteSaver використовується
💰 Gemini API не викликався


In [31]:
from langgraph.types import Command

# ============================================================
# HITL APPROVE DEMO — КРОК 1
# Запуск до interrupt()
# ============================================================

hitl_config_approve = {
    "configurable": {
        "thread_id": "hitl-approve-001"
    }
}

hitl_initial_state = {
    "hotel_name": "Paris Demo Hotel",
    "check_in": "2026-09-15",
    "nights": 3,
    "total_cost": 360.0,
    "result": "",
    "approved": False,
}

interrupt_result = app_hitl.invoke(
    hitl_initial_state,
    config=hitl_config_approve
)

print("⏸️ Граф зупинився перед ризиковою дією")
print("Thread ID:", hitl_config_approve["configurable"]["thread_id"])
print()
print("Результат invoke:")
print(interrupt_result)

print("\n💰 Gemini API не викликався")

⏸️ Граф зупинився перед ризиковою дією
Thread ID: hitl-approve-001

Результат invoke:
{'hotel_name': 'Paris Demo Hotel', 'check_in': '2026-09-15', 'nights': 3, 'total_cost': 360.0, 'result': '', 'approved': False, '__interrupt__': [Interrupt(value={'action': 'book_hotel', 'message': 'Підтвердіть бронювання готелю', 'details': {'hotel_name': 'Paris Demo Hotel', 'check_in': '2026-09-15', 'nights': 3, 'total_cost': 360.0}}, id='146a67d7a599d7ecef0598057ca95c17')]}

💰 Gemini API не викликався


In [32]:
Command(resume={"approved": True})

Command(resume={'approved': True})

In [33]:
# ============================================================
# HITL APPROVE DEMO — КРОК 2
# Продовження після підтвердження людини
# ============================================================

approved_result = app_hitl.invoke(
    Command(resume={"approved": True}),
    config=hitl_config_approve
)

print("✅ Користувач підтвердив ризикову дію")
print("Thread ID:", hitl_config_approve["configurable"]["thread_id"])

print("\nApproved:")
print(approved_result["approved"])

print("\nResult:")
print(approved_result["result"])

print("\n💰 Gemini API не викликався")

✅ Користувач підтвердив ризикову дію
Thread ID: hitl-approve-001

Approved:
True

Result:
ЗАБРОНЬОВАНО: Paris Demo Hotel, заїзд 2026-09-15, 3 ночей, вартість 360.00 EUR

💰 Gemini API не викликався


In [34]:
# ============================================================
# HITL REJECT DEMO — КРОК 1
# Окрема сесія
# ============================================================

hitl_config_reject = {
    "configurable": {
        "thread_id": "hitl-reject-001"
    }
}

reject_initial_state = {
    "hotel_name": "Paris Expensive Hotel",
    "check_in": "2026-09-20",
    "nights": 4,
    "total_cost": 1200.0,
    "result": "",
    "approved": False,
}

reject_interrupt = app_hitl.invoke(
    reject_initial_state,
    config=hitl_config_reject
)

print("⏸️ Reject-сесія зупинилась перед ризиковою дією")
print("Thread ID:", hitl_config_reject["configurable"]["thread_id"])
print()
print(reject_interrupt)

print("\n💰 Gemini API не викликався")

⏸️ Reject-сесія зупинилась перед ризиковою дією
Thread ID: hitl-reject-001

{'hotel_name': 'Paris Expensive Hotel', 'check_in': '2026-09-20', 'nights': 4, 'total_cost': 1200.0, 'result': '', 'approved': False, '__interrupt__': [Interrupt(value={'action': 'book_hotel', 'message': 'Підтвердіть бронювання готелю', 'details': {'hotel_name': 'Paris Expensive Hotel', 'check_in': '2026-09-20', 'nights': 4, 'total_cost': 1200.0}}, id='1f28fd9b61b56524467abcb45210ea27')]}

💰 Gemini API не викликався


In [35]:
# ============================================================
# HITL REJECT DEMO — КРОК 2
# Людина відхиляє дію
# ============================================================

rejected_result = app_hitl.invoke(
    Command(
        resume={
            "approved": False,
            "reason": "Занадто висока вартість"
        }
    ),
    config=hitl_config_reject
)

print("❌ Користувач відхилив ризикову дію")
print("Thread ID:", hitl_config_reject["configurable"]["thread_id"])

print("\nApproved:")
print(rejected_result["approved"])

print("\nResult:")
print(rejected_result["result"])

print("\n💰 Gemini API не викликався")

❌ Користувач відхилив ризикову дію
Thread ID: hitl-reject-001

Approved:
False

Result:
Бронювання ВІДХИЛЕНО. Причина: Занадто висока вартість

💰 Gemini API не викликався


In [36]:
def booking_with_approval(state: HITLState):
    """
    HITL: approve / reject / edit.
    """

    approval = interrupt({
        "action": "book_hotel",
        "message": "Підтвердіть бронювання готелю",
        "details": {
            "hotel_name": state["hotel_name"],
            "check_in": state["check_in"],
            "nights": state["nights"],
            "total_cost": state["total_cost"],
        }
    })

    # APPROVE
    if isinstance(approval, dict) and approval.get("approved") is True:

        result = book_hotel.invoke({
            "hotel_name": state["hotel_name"],
            "check_in": state["check_in"],
            "nights": state["nights"],
            "total_cost": state["total_cost"],
        })

        return {
            "approved": True,
            "result": result,
        }

    # EDIT
    if isinstance(approval, dict) and approval.get("action") == "edit":

        edited_args = approval.get("edited_args", {})

        hotel_name = edited_args.get(
            "hotel_name",
            state["hotel_name"]
        )

        check_in = edited_args.get(
            "check_in",
            state["check_in"]
        )

        nights = edited_args.get(
            "nights",
            state["nights"]
        )

        total_cost = edited_args.get(
            "total_cost",
            state["total_cost"]
        )

        result = book_hotel.invoke({
            "hotel_name": hotel_name,
            "check_in": check_in,
            "nights": nights,
            "total_cost": total_cost,
        })

        return {
            "approved": True,
            "result": (
                "Параметри змінено оператором. "
                + result
            ),
        }

    # REJECT
    reason = ""

    if isinstance(approval, dict):
        reason = approval.get("reason", "")

    return {
        "approved": False,
        "result": f"Бронювання ВІДХИЛЕНО. Причина: {reason}",
    }


print("✅ HITL node оновлено: approve / reject / edit")
print("💰 Gemini API не викликався")

✅ HITL node оновлено: approve / reject / edit
💰 Gemini API не викликався


In [37]:
hitl_graph_v2 = StateGraph(HITLState)

hitl_graph_v2.add_node(
    "booking_with_approval",
    booking_with_approval
)

hitl_graph_v2.add_edge(
    START,
    "booking_with_approval"
)

hitl_graph_v2.add_edge(
    "booking_with_approval",
    END
)

app_hitl_v2 = hitl_graph_v2.compile(
    checkpointer=new_saver
)

print("✅ HITL graph v2 готовий")
print("💰 Gemini API не викликався")

✅ HITL graph v2 готовий
💰 Gemini API не викликався


In [38]:
# ============================================================
# HITL EDIT DEMO — КРОК 1
# ============================================================

hitl_config_edit = {
    "configurable": {
        "thread_id": "hitl-edit-001"
    }
}

edit_initial_state = {
    "hotel_name": "Paris Demo Hotel",
    "check_in": "2026-09-25",
    "nights": 4,
    "total_cost": 500.0,
    "result": "",
    "approved": False,
}

edit_interrupt = app_hitl_v2.invoke(
    edit_initial_state,
    config=hitl_config_edit
)

print("⏸️ Edit-сесія зупинилась перед ризиковою дією")
print("Thread ID:", hitl_config_edit["configurable"]["thread_id"])
print()
print(edit_interrupt)

print("\n💰 Gemini API не викликався")

⏸️ Edit-сесія зупинилась перед ризиковою дією
Thread ID: hitl-edit-001

{'hotel_name': 'Paris Demo Hotel', 'check_in': '2026-09-25', 'nights': 4, 'total_cost': 500.0, 'result': '', 'approved': False, '__interrupt__': [Interrupt(value={'action': 'book_hotel', 'message': 'Підтвердіть бронювання готелю', 'details': {'hotel_name': 'Paris Demo Hotel', 'check_in': '2026-09-25', 'nights': 4, 'total_cost': 500.0}}, id='5e57e5350e1535634ae2de6b42ab7cf7')]}

💰 Gemini API не викликався


In [39]:
# ============================================================
# HITL EDIT DEMO — КРОК 2
# Змінюємо параметри перед виконанням
# ============================================================

edited_result = app_hitl_v2.invoke(
    Command(
        resume={
            "action": "edit",
            "edited_args": {
                "hotel_name": "Paris Demo Hotel",
                "check_in": "2026-09-25",
                "nights": 3,
                "total_cost": 320.0
            }
        }
    ),
    config=hitl_config_edit
)

print("✏️ Користувач змінив параметри ризикової дії")
print("Thread ID:", hitl_config_edit["configurable"]["thread_id"])

print("\nApproved:")
print(edited_result["approved"])

print("\nResult:")
print(edited_result["result"])

print("\n💰 Gemini API не викликався")

✏️ Користувач змінив параметри ризикової дії
Thread ID: hitl-edit-001

Approved:
True

Result:
Параметри змінено оператором. ЗАБРОНЬОВАНО: Paris Demo Hotel, заїзд 2026-09-25, 3 ночей, вартість 320.00 EUR

💰 Gemini API не викликався


In [40]:
import os

print("=== ПЕРЕВІРКА АРТЕФАКТІВ ===")

print("agent_state.db існує:", os.path.exists("agent_state.db"))
if os.path.exists("agent_state.db"):
    print("Розмір agent_state.db:", os.path.getsize("agent_state.db"), "bytes")

print()

print("chroma_db існує:", os.path.exists("chroma_db"))

if os.path.exists("chroma_db"):
    print("Файли ChromaDB:")
    for root, dirs, files in os.walk("chroma_db"):
        for file in files:
            print(" -", os.path.join(root, file))

print("\n✅ Перевірка завершена")
print("💰 Gemini API не викликався")

=== ПЕРЕВІРКА АРТЕФАКТІВ ===
agent_state.db існує: True
Розмір agent_state.db: 28672 bytes

chroma_db існує: True
Файли ChromaDB:
 - chroma_db/chroma.sqlite3
 - chroma_db/1d485bc1-c407-434e-8cb3-d1636a108177/header.bin
 - chroma_db/1d485bc1-c407-434e-8cb3-d1636a108177/length.bin
 - chroma_db/1d485bc1-c407-434e-8cb3-d1636a108177/link_lists.bin
 - chroma_db/1d485bc1-c407-434e-8cb3-d1636a108177/data_level0.bin

✅ Перевірка завершена
💰 Gemini API не викликався


In [42]:
%%writefile README.md
# Домашнє завдання №2
## Plan-and-Execute з Memory, Agentic RAG та Human-in-the-Loop

Автор: Антон Бабенко

---

## 1. Опис проєкту

У роботі реалізовано AI-агента з архітектурою Plan-and-Execute на базі LangGraph.

Агент спочатку формує повний план виконання задачі, після чого виконує його послідовно. Після кожного виконаного кроку replanner оцінює актуальність плану та приймає рішення: продовжити виконання, перепланувати залишкові кроки або завершити задачу.

Реалізовано:

1. Plan-and-Execute: planner → executor → replanner.
2. Persistence через SqliteSaver.
3. Agentic RAG через ChromaDB.
4. Human-in-the-Loop через interrupt()/Command(resume=...).

---

## 2. Використані технології

- Python
- Google Colab
- LangGraph
- LangChain
- Gemini API
- Pydantic
- SQLite / SqliteSaver
- ChromaDB

---

## 3. Архітектура Plan-and-Execute

Основний граф:

START → planner → executor → replanner → executor / END

### Planner

Planner отримує запит користувача та формує структурований план.

Використовується Pydantic-модель `Plan`:

- `goal`
- `steps`

Structured output:

`planner_llm = llm.with_structured_output(Plan)`

### Executor

Executor виконує один крок плану за одну ітерацію.

Доступні tools:

- calculator
- get_weather
- convert_currency
- search_hotels
- search_knowledge

### Replanner

Replanner аналізує результати виконаних кроків та повертає `ReplanDecision`.

Можливі рішення:

- continue
- replan
- finish

---

## 4. Демонстрація Plan-and-Execute

Тестова задача:

Порахуй 120 * 3 + 45 і потім конвертуй результат за курсом 1.08.

Planner сформував два кроки:

1. Використати calculator.
2. Використати convert_currency.

Результати:

- Крок 1: calculator: 405
- Крок 2: convert_currency: 405.0 × 1.08 = 437.40
- Completed: True

---

## 5. Persistence та SqliteSaver

Для persistence використовується файловий SQLite checkpointer:

`agent_state.db`

З'єднання створюється через `sqlite3.connect()` та передається у `SqliteSaver`.

Граф компілюється з checkpointer, тому стан зберігається після виконання вузлів.

---

## 6. Відновлення стану

Тестова задача:

Порахуй 50 * 4, а потім конвертуй результат за курсом 1.1.

Граф було зупинено після першого executor.

На момент зупинки:

- Thread ID: persistence-demo-001
- Current step: 1
- Крок 1: calculator: 200
- Completed: False

Після закриття старого SQLite-з'єднання було створено нове з'єднання до того самого `agent_state.db`.

Стан відновився з того самого `thread_id`.

Після продовження:

- Крок 1: calculator: 200
- Крок 2: convert_currency: 200.0 × 1.1 = 220.00
- Completed: True

Таким чином агент продовжив виконання з checkpoint, а не почав задачу заново.

---

## 7. Незалежність thread_id

Продемонстровано дві сесії:

- persistence-demo-001
- persistence-demo-002

Перша сесія містила завершений стан.

Друга сесія з новим thread_id мала порожній стан.

Це демонструє незалежність різних сесій.

---

## 8. Agentic RAG

Для локальної бази знань використано ChromaDB.

База створена через PersistentClient та зберігається у директорії:

`chroma_db`

У базу завантажено 10 документів туристичної тематики.

Тематика документів:

- правила перебування у Шенгенській зоні;
- Париж;
- транспорт;
- Лувр;
- квитки;
- сезони;
- безпека;
- бюджет;
- готелі;
- страхування.

---

## 9. search_knowledge

Створено LangChain tool:

`search_knowledge`

Tool повертає топ-3 релевантних документи.

Для покращення українськомовного пошуку реалізовано hybrid search:

1. semantic search через ChromaDB;
2. keyword reranking.

На запит про правила перебування українців у Шенгенській зоні першим повертається документ про:

- біометричний паспорт;
- безвіз;
- правило 90 днів протягом 180-денного періоду.

---

## 10. Автономний вибір RAG

Агент самостійно вибирає потрібний tool.

Для запиту про правила короткострокового перебування громадян України у Шенгенській зоні агент вибрав:

`search_knowledge`

Для математичного запиту:

`Порахуй 250 * 4 + 100`

агент вибрав:

`calculator`

Результат:

`1100`

Таким чином продемонстровано Agentic RAG: база знань використовується тільки тоді, коли вона справді потрібна.

---

## 11. Human-in-the-Loop

Для HITL створено ризиковий tool:

`book_hotel`

Перед виконанням бронювання викликається `interrupt()`.

Граф зупиняється та показує:

- назву готелю;
- дату заїзду;
- кількість ночей;
- загальну вартість.

Для HITL використовується SqliteSaver.

---

## 12. Approve flow

Початкові параметри:

- Paris Demo Hotel
- 2026-09-15
- 3 ночі
- 360 EUR

Після `Command(resume={"approved": True})`:

- Approved: True
- бронювання виконано.

Результат:

ЗАБРОНЬОВАНО: Paris Demo Hotel, заїзд 2026-09-15, 3 ночей, вартість 360.00 EUR

---

## 13. Reject flow

Окрема HITL-сесія:

- Paris Expensive Hotel
- 4 ночі
- 1200 EUR

Людина відхилила дію з причиною:

`Занадто висока вартість`

Результат:

- Approved: False
- бронювання не виконувалося.

---

## 14. Edit flow

Додатково реалізовано зміну параметрів перед виконанням.

Початково:

- 4 ночі
- 500 EUR

Після edit:

- 3 ночі
- 320 EUR

Результат:

Параметри змінено оператором. ЗАБРОНЬОВАНО: Paris Demo Hotel, заїзд 2026-09-25, 3 ночей, вартість 320.00 EUR

---

## 15. Аналіз результатів

Plan-and-Execute дозволяє розбивати складні задачі на послідовність кроків та контролювати виконання кожного кроку.

SqliteSaver забезпечує persistence та дозволяє відновити роботу після перезапуску процесу.

Thread ID дозволяє підтримувати незалежні сесії.

Agentic RAG дозволяє LLM самостійно вирішувати, коли потрібна база знань, а коли треба використати інший tool.

HITL дозволяє зупиняти виконання перед ризиковими діями та передавати остаточне рішення людині.

У роботі також мінімізовано кількість LLM-запитів: локальні перевірки, persistence, ChromaDB та HITL виконуються без Gemini.

---

## 16. Запуск

Встановлення залежностей:

pip install langgraph langgraph-checkpoint-sqlite langchain langchain-core langchain-google-genai chromadb "pydantic>=2.0"

У Google Colab API-ключ Gemini зберігається у Secret:

`GOOGLE_API_KEY`

Notebook виконується послідовно зверху вниз.

---

## 17. Артефакти

До роботи входять:

- HW2_Babenko_Plan_Execute.ipynb
- README.md
- agent_state.db
- chroma_db/

---

## Висновок

У роботі реалізовано:

- Plan-and-Execute граф;
- planner;
- executor;
- replanner;
- structured outputs;
- 4+ tools;
- SqliteSaver;
- persistence;
- відновлення після restart;
- незалежні thread_id;
- ChromaDB з 10 документами;
- search_knowledge;
- Agentic RAG;
- HITL interrupt;
- approve;
- reject;
- edit.

Writing README.md


In [43]:
import os

print("README.md існує:", os.path.exists("README.md"))
print("Розмір:", os.path.getsize("README.md"), "bytes")

README.md існує: True
Розмір: 9499 bytes


In [44]:
import os
import shutil

ZIP_NAME = "HW2_Babenko"

# Створюємо тимчасову папку
os.makedirs(ZIP_NAME, exist_ok=True)

# Копіюємо README
shutil.copy("README.md", f"{ZIP_NAME}/README.md")

# Копіюємо SQLite
shutil.copy("agent_state.db", f"{ZIP_NAME}/agent_state.db")

# Копіюємо ChromaDB
if os.path.exists(f"{ZIP_NAME}/chroma_db"):
    shutil.rmtree(f"{ZIP_NAME}/chroma_db")

shutil.copytree(
    "chroma_db",
    f"{ZIP_NAME}/chroma_db"
)

# Створюємо архів
shutil.make_archive(
    ZIP_NAME,
    "zip",
    ZIP_NAME
)

print("✅ Архів створено:")
print(f"{ZIP_NAME}.zip")
print("Розмір:", os.path.getsize(f"{ZIP_NAME}.zip"), "bytes")

✅ Архів створено:
HW2_Babenko.zip
Розмір: 48739 bytes
